# Exercise 2.2 — Simple Expense Categoriser
### *Turning "Grab" into "Transport" without doing it 60 times by hand*

If you've ever exported a bank statement and stared at a wall of cryptic merchant names, you know the pain: `GRAB *RIDE`, `NTUC FairPrice`, `SP Group Utilities`... none of it means anything until you mentally sort it into categories. Let's teach the computer to do that sorting.

**Why learn this instead of uploading your bank statement to an AI tool?**

Because bank statements are sensitive financial data, and once you understand this pattern, you can categorise your expenses **locally, offline, for free, forever** — without uploading a single transaction anywhere. And the underlying idea — "match a keyword, assign a label" — is the exact same logic behind spam filters, support-ticket routing, and content moderation systems.

## What you'll practise
- **Dictionaries** — mapping keywords to categories
- **File reading** — working through a real CSV of transactions
- **Conditionals** — deciding which category a transaction belongs to

## The scenario
`transactions.csv` contains 60 fictional bank transactions from May 2026. Let's tag each one automatically.

## Step 1: Load and inspect the transactions

In [1]:
import csv

with open("transactions.csv", newline="") as f:
    reader = csv.DictReader(f)
    transactions = list(reader)

print(f"Loaded {len(transactions)} transactions")
for t in transactions[:5]:
    print(t)


Loaded 60 transactions
{'date': '2026-05-02', 'description': 'SMRT Fare', 'amount': '-126.61'}
{'date': '2026-05-02', 'description': 'Lazada', 'amount': '-63.61'}
{'date': '2026-05-02', 'description': 'Shopee', 'amount': '-167.52'}
{'date': '2026-05-02', 'description': 'SBS Transit', 'amount': '-39.32'}
{'date': '2026-05-03', 'description': 'GRAB *RIDE', 'amount': '-97.78'}


Notice `amount` came in as **text** (e.g. `"-12.5"`), not a number — CSV files don't know about number types, everything is a string until you convert it. Let's fix that.

In [2]:
for t in transactions:
    t["amount"] = float(t["amount"])

print(transactions[0])
print(type(transactions[0]["amount"]))


{'date': '2026-05-02', 'description': 'SMRT Fare', 'amount': -126.61}
<class 'float'>


## Step 2: Define our categories as a dictionary of keywords

In [3]:
# Each category maps to a list of keywords we'll look for in the description.
# Keep keywords lowercase - we'll lowercase the description before comparing.
category_keywords = {
    "Transport": ["grab", "comfortdelgro", "taxi", "smrt", "sbs transit"],
    "Groceries": ["ntuc", "fairprice", "cold storage"],
    "Food & Drink": ["starbucks", "toast box", "old chang kee"],
    "Utilities & Bills": ["sp group", "singtel"],
    "Subscriptions": ["netflix", "spotify"],
    "Shopping": ["shopee", "lazada", "amazon", "popular bookstore"],
    "Health": ["guardian", "watsons"],
    "Income": ["client payment", "salary"],
}

print("Categories defined:", list(category_keywords.keys()))


Categories defined: ['Transport', 'Groceries', 'Food & Drink', 'Utilities & Bills', 'Subscriptions', 'Shopping', 'Health', 'Income']


## Step 3: Write a function that categorises one transaction

In [4]:
def categorise(description):
    description_lower = description.lower()

    for category, keywords in category_keywords.items():
        for keyword in keywords:
            if keyword in description_lower:
                return category

    return "Uncategorised"

# quick tests
print(categorise("GRAB *RIDE"))
print(categorise("NTUC FairPrice"))
print(categorise("Some Random Shop"))


Transport
Groceries
Uncategorised


**What just happened?** This function has a loop *inside* a loop: for each category, check each of its keywords, and see `if keyword in description_lower`. The `in` check here is doing **substring matching** — `"grab" in "grab *ride"` is `True` because "grab" appears somewhere inside that longer string. The moment we find a match, `return` immediately exits the function with the answer — we don't need to keep checking other categories.

## Step 4: Apply it to every transaction

In [5]:
for t in transactions:
    t["category"] = categorise(t["description"])

for t in transactions[:10]:
    print(f"{t['date']}  {t['description']:<25} {t['amount']:>8.2f}  ->  {t['category']}")


2026-05-02  SMRT Fare                  -126.61  ->  Transport
2026-05-02  Lazada                      -63.61  ->  Shopping
2026-05-02  Shopee                     -167.52  ->  Shopping
2026-05-02  SBS Transit                 -39.32  ->  Transport
2026-05-03  GRAB *RIDE                  -97.78  ->  Transport
2026-05-04  Grab                        -84.24  ->  Transport
2026-05-04  Salary Payout               734.41  ->  Income
2026-05-04  Lazada                     -149.57  ->  Shopping
2026-05-04  SP Group Utilities           -6.22  ->  Utilities & Bills
2026-05-04  Netflix.com                 -33.33  ->  Subscriptions


## Step 5: Summarise spending by category

In [6]:
category_totals = {}

for t in transactions:
    category = t["category"]
    amount = t["amount"]
    category_totals[category] = category_totals.get(category, 0) + amount

print(f"{'CATEGORY':<20}{'TOTAL':>10}")
print("-" * 30)
for category, total in sorted(category_totals.items(), key=lambda pair: pair[1]):
    print(f"{category:<20}{total:>10.2f}")


CATEGORY                 TOTAL
------------------------------
Shopping              -1097.69
Transport              -916.76
Food & Drink           -666.82
Groceries              -587.16
Health                 -540.49
Utilities & Bills      -380.11
Subscriptions           -70.18
Income                19337.52


**Note the `.get(category, 0)`** — this is a very handy dictionary trick. `category_totals.get(category, 0)` means "give me the current total for this category, or 0 if we haven't seen it yet." Without it, we'd get an error the first time we hit a brand-new category, because it wouldn't exist in the dictionary yet.

## 🎉 You did it

You just built a working expense categoriser using nothing but a dictionary of keywords and a couple of loops. No AI subscription, no data leaving your laptop, and it runs in milliseconds.

---

## 🧪 Try it yourself

1. Add a new category of your own (e.g. "Entertainment") with a few keywords, and re-run.
2. Some transactions probably ended up as "Uncategorised" — print them out and figure out which keywords are missing.
3. Instead of just totals, calculate what **percentage** of total spending each category represents.

## 💡 Where this goes next
This "match keyword → assign category" pattern, scaled up with smarter matching, is genuinely how a lot of budgeting apps work under the hood — and it's also the starting point for building simple rule-based classifiers before you ever need a fancier AI model.